# EXP4: Technique Family Classification

## Centralized imports
Imports all required libraries. Run this cell first.

In [ ]:
# Centralized imports

import os
import sys
import io
import json
import glob
import shutil
import time
import warnings
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


## Cell 0 — Path Configuration
Central path registry for the entire notebook.
Every downstream cell reads paths from the `paths` object defined here — no hardcoded
directories anywhere else. Update **only this cell** when migrating between machines.


In [ ]:
# Cell 0 — Path Configuration (EXP4 — Technique Family Classification)

@dataclass
class ExpPaths:
    """Central path registry for EXP4."""

    base:       str = os.getenv("SECD_BASE", str(Path.cwd()))
    cache_root: str = ""
    outdir:     str = ""
    split_dir:  str = ""
    csv_dirs:   list = field(default_factory=list)

    def __post_init__(self):
        self.cache_root = os.path.join(self.base, "mel_cache_ast16k_256")
        self.outdir     = os.path.join(self.base, "EXP4_TEC_FAM")
        self.split_dir  = os.path.join(self.outdir, "exp4_splits")

        self.csv_dirs = []
        for group in [
            "harmonic_intervals_loose", "harmonic_intervals_strict",
            "triads_loose", "triads_strict",
            "7th_chords_loose", "7th_chords_strict",
        ]:
            d = os.path.join(self.base, group, "csv")
            if os.path.isdir(d):
                self.csv_dirs.append(d)

    def verify(self):
        errors = []
        if not os.path.isdir(self.cache_root):
            errors.append(f"Cache not found: {self.cache_root}")
        if not self.csv_dirs:
            errors.append("No CSV directories found")
        for d in self.csv_dirs:
            csvs = [f for f in os.listdir(d)
                    if f.endswith(".csv") and "-original" not in f]
            if not csvs:
                errors.append(f"No usable CSVs in {d}")
        if errors:
            for e in errors:
                print(f"  {e}")
            raise RuntimeError("Path verification failed")

        os.makedirs(self.outdir, exist_ok=True)
        os.makedirs(self.split_dir, exist_ok=True)

        all_csvs = []
        for d in self.csv_dirs:
            all_csvs += [os.path.join(d, f) for f in os.listdir(d)
                         if f.endswith(".csv") and "-original" not in f]

        print(f"  Base       : {self.base}")
        print(f"  Cache      : {self.cache_root}")
        print(f"  Output     : {self.outdir}")
        print(f"  Splits     : {self.split_dir}")
        print(f"  CSV dirs   : {len(self.csv_dirs)}")
        print(f"  CSV files  : {len(all_csvs)} (excluding -original)")

        return all_csvs

paths = ExpPaths()
all_csv_paths = paths.verify()
print(f"\nEXP4 Path configuration OK")


## Cell 3 — Constants, Paths, CSV Inventory (EXP4: technique families)
Establishes all experiment-level constants for EXP4.
EXP4 classifies **global technique family** (bowed / plucked / col_legno / harmonic).
Uses strict and loose subsets as metadata sources for this task.


In [ ]:
# Cell 3 — Constants, paths, CSV inventory, validated technique-family setup (EXP4)

# ── AST model ──
MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
TARGET_FRAMES = 256

# ── Instruments ──
INSTR_ORDER = ["cello", "viola", "violin2", "violin1"]
INSTRUMENTS = INSTR_ORDER

# ── Full validated global technique vocabulary (SECD-wide across inspected CSVs) ──
TECHNIQUES = [
    "arco-normal", "arco-sul-ponticello", "arco-sul-tasto", "arco-au-talon",
    "arco-tremolo", "arco-martele", "arco-minor-trill", "arco-major-trill",
    "arco-glissando", "arco-col-legno-battuto", "arco-col-legno-tratto",
    "arco-harmonic",
    "pizz-normal", "pizz-glissando", "pizz-tremolo", "snap-pizz",
    "natural-harmonic", "artificial-harmonic",
    "molto-vibrato", "non-vibrato", "con-sord",
]

# ── Dynamics vocabulary (kept for data loading) ──
DYNAMICS = [
    "pianissimo", "piano", "mezzo-piano", "mezzo-forte", "forte", "fortissimo"
]

# ── Technique family mapping ──
TEC_TO_FAMILY = {
    "arco-normal":             "bowed",
    "arco-sul-ponticello":     "bowed",
    "arco-sul-tasto":          "bowed",
    "arco-au-talon":           "bowed",
    "arco-tremolo":            "bowed",
    "arco-martele":            "bowed",
    "arco-minor-trill":        "bowed",
    "arco-major-trill":        "bowed",
    "arco-glissando":          "bowed",
    "molto-vibrato":           "bowed",
    "non-vibrato":             "bowed",
    "con-sord":                "bowed",
    "pizz-normal":             "plucked",
    "pizz-glissando":          "plucked",
    "pizz-tremolo":            "plucked",
    "snap-pizz":               "plucked",
    "arco-col-legno-battuto":  "col_legno",
    "arco-col-legno-tratto":   "col_legno",
    "natural-harmonic":        "harmonic",
    "artificial-harmonic":     "harmonic",
    "arco-harmonic":           "harmonic",
}

TECHNIQUE_FAMILIES = sorted(set(TEC_TO_FAMILY.values()))

# ── Mel cache ──
CACHE_ROOT = paths.cache_root

# ── CSV helpers ──
CANDIDATE_FILENAME_COLS = [
    "chord_filename", "filename", "wav_filename", "output_wav", "wav_file"
]

def resolve_filename_column(df):
    for c in CANDIDATE_FILENAME_COLS:
        if c in df.columns:
            return c
    raise KeyError("Filename column not found in CSV.")

def derive_audio_root_from_csv(csv_path: str) -> str:
    csv_dir, csv_file = os.path.split(csv_path)
    base = os.path.splitext(csv_file)[0]
    wav_dir = csv_dir.replace("/csv", "/wav")
    return os.path.join(wav_dir, base) + "/"

def _cand_dyn_cols(inst: str):
    return [f"{inst}_dynamic", f"{inst}_dyn", f"dynamic_{inst}", f"dyn_{inst}"]

def _cand_tec_cols(inst: str):
    return [f"{inst}_technique", f"{inst}_tec", f"technique_{inst}", f"tec_{inst}"]

# ── CSV inventory ──
_CSV_MAP = {
    "duos_loose": ("harmonic_intervals_loose", [
        "cello_viola_loose.csv", "cello_violin_loose.csv", "viola_violin_loose.csv"]),
    "trios_loose": ("triads_loose", [
        "triads_major_loose.csv", "triads_minor_loose.csv",
        "triads_diminished_loose.csv", "triads_augmented_loose.csv"]),
    "quartets_loose": ("7th_chords_loose", [
        "major_seventh_loose.csv", "minor_seventh_loose.csv",
        "dominant_seventh_loose.csv", "half_diminished_seventh_loose.csv"]),
    "duos_strict": ("harmonic_intervals_strict", [
        "cello_viola_strict.csv", "cello_violin_strict.csv", "viola_violin_strict.csv"]),
    "trios_strict": ("triads_strict", [
        "triads_major_strict.csv", "triads_minor_strict.csv",
        "triads_diminished_strict.csv", "triads_augmented_strict.csv"]),
    "quartets_strict": ("7th_chords_strict", [
        "major_seventh_strict.csv", "minor_seventh_strict.csv",
        "dominant_seventh_strict.csv", "half_diminished_seventh_strict.csv"]),
}

def _build_csv_list(group_key):
    folder, files = _CSV_MAP[group_key]
    return [os.path.join(paths.base, folder, "csv", f) for f in files]

csv_duos_loose      = _build_csv_list("duos_loose")
csv_trios_loose     = _build_csv_list("trios_loose")
csv_quartets_loose  = _build_csv_list("quartets_loose")
csv_duos_strict     = _build_csv_list("duos_strict")
csv_trios_strict    = _build_csv_list("trios_strict")
csv_quartets_strict = _build_csv_list("quartets_strict")

ALL_CSVS = (
    csv_duos_loose + csv_trios_loose + csv_quartets_loose +
    csv_duos_strict + csv_trios_strict + csv_quartets_strict
)

_missing = [p for p in ALL_CSVS if not os.path.isfile(p)]
if _missing:
    print("Missing CSV files:")
    for p in _missing:
        print("  -", p)
    raise FileNotFoundError(f"{len(_missing)} CSV(s) missing.")

# ── Validation against discovered schema from Cell 2b ──
if "TECHNIQUE_SCHEMA_REGISTRY" not in globals():
    raise RuntimeError("Cell 2b must be executed before Cell 3.")

if "get_row_techniques" not in globals():
    raise RuntimeError("Cell 2c must be executed before Cell 3.")

schema_types_seen = set()
globally_observed_text_techniques = set()

for csv_path in ALL_CSVS:
    if csv_path not in TECHNIQUE_SCHEMA_REGISTRY:
        raise RuntimeError(f"Missing schema entry for: {csv_path}")

    info = TECHNIQUE_SCHEMA_REGISTRY[csv_path]
    schema_types_seen.add(info["schema_type"])
    globally_observed_text_techniques.update(info["observed_text_techniques"])

unknown_observed = sorted(globally_observed_text_techniques - set(TECHNIQUES))
unmapped_declared = sorted(set(TECHNIQUES) - set(TEC_TO_FAMILY.keys()))

if unknown_observed:
    print("Observed techniques missing from TECHNIQUES:")
    for x in unknown_observed:
        print("  -", x)
    raise RuntimeError("Cell 3 vocabulary is incomplete.")

if unmapped_declared:
    print("Declared techniques missing from TEC_TO_FAMILY:")
    for x in unmapped_declared:
        print("  -", x)
    raise RuntimeError("Technique family mapping is incomplete.")

print(f"  Technique families : {TECHNIQUE_FAMILIES}")
print("  Family sizes       :")
for fam in TECHNIQUE_FAMILIES:
    members = [t for t, f in TEC_TO_FAMILY.items() if f == fam]
    print(f"    {fam:12s} : {len(members):2d} techniques — {members}")

print("\n  Validated metadata schema:")
print(f"    schema types seen     : {sorted(schema_types_seen)}")
print("    duo handling          : per-instrument techniques derived from source filenames")
print("    trio/quartet handling : per-instrument technique columns used directly")

print(f"\n   CSV inventory OK — {len(ALL_CSVS)} files found.")
print(f"   FAMILIES    : {len(TECHNIQUE_FAMILIES)} classes — {TECHNIQUE_FAMILIES}")
print(f"   TECHNIQUES  : {len(TECHNIQUES)} raw techniques")
print(f"   OBSERVED    : {len(globally_observed_text_techniques)} validated textual techniques")
print(f"   CACHE_ROOT  : {CACHE_ROOT}")
print(f"   OUTDIR      : {paths.outdir}")


## Cell 4 — Load Metadata & Map Technique Families (EXP4)
Loads SECD metadata and maps per-instrument technique labels to the coarse
technique-family targets used by EXP4. All valid samples are retained for
this global technique-family classification setup.


In [ ]:
# Cell 4 — Load metadata & map technique families (ALL samples, no filter)

def _abs_path(audio_root: str, x: str) -> str:
    x = str(x).strip()
    return x if os.path.isabs(x) else os.path.normpath(os.path.join(audio_root, x))

def _cache_path(cache_root: str, csv_path: str, abs_wav_path: str) -> str:
    """Derive cache .npy path from the CSV path and WAV filename.

    Cache structure mirrors the WAV directory layout:
      {cache_root}/{group}/{csv_stem}/{wav_stem}.npy
    """
    csv_dir = os.path.dirname(csv_path)
    group = os.path.basename(os.path.dirname(csv_dir))
    csv_stem = os.path.splitext(os.path.basename(csv_path))[0]
    wav_stem = os.path.splitext(os.path.basename(abs_wav_path))[0]
    return os.path.join(cache_root, group, csv_stem, f"{wav_stem}.npy")

def load_block(csv_list, ensemble_tag, cache_root):
    """Load metadata from a list of CSVs and create per-sample entries."""
    frames = []

    for csv_path in csv_list:
        if not os.path.exists(csv_path):
            print(f"   Warning: CSV not found, skipping: {csv_path}")
            continue

        df = pd.read_csv(csv_path)
        fname_col = resolve_filename_column(df)
        audio_root = derive_audio_root_from_csv(csv_path)
        subset = "strict" if "strict" in csv_path else "loose"

        for _, row in df.iterrows():
            abs_fp = _abs_path(audio_root, row[fname_col])
            cf = _cache_path(cache_root, csv_path, abs_fp)

            techs = get_row_techniques(row, csv_path=csv_path)
            dyns = get_row_dynamics(row, csv_path=csv_path)

            entry = {
                "filepath": abs_fp,
                "cachefile": cf,
                "ensemble_size": ensemble_tag,
                "subset": subset,
                "csv_source": os.path.basename(csv_path),
            }

            for inst in INSTR_ORDER:
                tec_val = techs.get(inst, None)
                dyn_val = dyns.get(inst, None)

                entry[f"tec_{inst}"] = tec_val
                entry[f"dyn_{inst}"] = dyn_val
                entry[f"has_{inst}"] = (tec_val is not None) or (dyn_val is not None)

            frames.append(entry)

    return pd.DataFrame(frames)

if "get_row_techniques" not in globals():
    raise RuntimeError("Cell 2c must be executed before Cell 4.")

if "get_row_dynamics" not in globals():
    raise RuntimeError("Cell 2c dynamic helpers must be defined before Cell 4.")

print("Loading all SECD metadata...")

blocks = []
for tag, csvs in [
    ("duo",     csv_duos_loose),
    ("trio",    csv_trios_loose),
    ("quartet", csv_quartets_loose),
    ("duo",     csv_duos_strict),
    ("trio",    csv_trios_strict),
    ("quartet", csv_quartets_strict),
]:
    block = load_block(csvs, tag, CACHE_ROOT)
    subset = "strict" if "strict" in csvs[0] else "loose"
    blocks.append(block)
    print(f"   {subset:6s} {tag:8s} : {len(block):>7,} samples")

df_all = pd.concat(blocks, ignore_index=True)
print(f"\n   Total samples loaded: {len(df_all):,}")

# ── Map techniques → families ──
n_mapped = 0
n_unmapped = 0

for inst in INSTR_ORDER:
    tec_col = f"tec_{inst}"
    fam_col = f"fam_{inst}"

    families = []
    for tec in df_all[tec_col]:
        if pd.isna(tec) or tec is None:
            families.append(None)
        elif str(tec) in TEC_TO_FAMILY:
            families.append(TEC_TO_FAMILY[str(tec)])
            n_mapped += 1
        else:
            families.append(None)
            n_unmapped += 1

    df_all[fam_col] = families

print(f"\n   Technique → Family mapping:")
print(f"     Mapped   : {n_mapped:>9,} labels")
print(f"     Unmapped : {n_unmapped:>9,} labels (instrument absent or unknown)")

# ── Verify cache existence ──
cache_exists = df_all["cachefile"].apply(os.path.exists)
n_cached = int(cache_exists.sum())
n_missing_cache = int((~cache_exists).sum())

print(f"\n   Cache files found   : {n_cached:>9,}")
print(f"   Cache files missing : {n_missing_cache:>9,}")

if n_missing_cache > 0 and n_cached == 0:
    sample_cf = df_all["cachefile"].iloc[0]
    print(f"\n   Sample expected cache path:")
    print(f"     {sample_cf}")
    print(f"     Exists: {os.path.exists(sample_cf)}")
    if os.path.isdir(CACHE_ROOT):
        subdirs = sorted(os.listdir(CACHE_ROOT))[:10]
        print(f"     Cache root contains: {subdirs}")
    raise RuntimeError(
        "All cache files missing. Check _cache_path() logic versus actual cache structure."
    )
elif n_missing_cache > 0:
    print(f"   Warning: {n_missing_cache:,} samples have no cached spectrogram — filtering them out.")
    df_all = df_all[cache_exists].reset_index(drop=True)

print(f"\n   Final df_all size: {len(df_all):,}")

# ── Quick coverage summary ──
print("\n   Non-null technique labels per instrument:")
for inst in INSTR_ORDER:
    n = int(df_all[f"tec_{inst}"].notna().sum())
    print(f"     {inst:8s}: {n:>9,}")

print("\n   Non-null family labels per instrument:")
for inst in INSTR_ORDER:
    n = int(df_all[f"fam_{inst}"].notna().sum())
    print(f"     {inst:8s}: {n:>9,}")

print("\ndf_all ready — all samples loaded with unified per-instrument technique families")


## Cell 5 — Stratified Train / Val / Test Split (70 / 15 / 15)
Stratifies by technique-family multi-label structure to preserve label balance across splits. Saves indices to disk.


In [ ]:
# Cell 5 — Exact-size stratified Train / Val / Test Split (70 / 15 / 15) — EXP4

OUT_SPLIT_DIR = Path(paths.split_dir)
OUT_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

_split_file    = OUT_SPLIT_DIR / "exp4_split_indices.npz"
_manifest_file = OUT_SPLIT_DIR / "exp4_manifest.csv"

def _hamilton_allocate(counts: np.ndarray, target_total: int) -> np.ndarray:
    """
    Exact integer allocation using Hamilton / largest remainder method.
    Guarantees:
      - sum(allocation) == target_total
      - 0 <= allocation[i] <= counts[i]
    """
    counts = np.asarray(counts, dtype=np.int64)
    total = int(counts.sum())

    if target_total < 0 or target_total > total:
        raise ValueError(f"Invalid target_total={target_total} for total={total}")

    if total == 0:
        return np.zeros_like(counts, dtype=np.int64)

    quotas = counts * (target_total / total)
    alloc = np.floor(quotas).astype(np.int64)

    # Safety cap
    alloc = np.minimum(alloc, counts)

    deficit = int(target_total - alloc.sum())
    if deficit > 0:
        remainders = quotas - alloc
        order = np.argsort(-remainders)  # descending
        for i in order:
            if deficit == 0:
                break
            if alloc[i] < counts[i]:
                alloc[i] += 1
                deficit -= 1

    if alloc.sum() != target_total:
        raise RuntimeError(
            f"Hamilton allocation failed: got {alloc.sum()} instead of {target_total}"
        )

    return alloc

def _build_strata_keys(df: pd.DataFrame) -> pd.Series:
    """
    Composite stratification key from ensemble size + per-instrument family labels.
    """
    fam_cols = [f"fam_{inst}" for inst in INSTR_ORDER]

    temp = pd.DataFrame(index=df.index)
    temp["ensemble_size"] = df["ensemble_size"].astype(str)

    for c in fam_cols:
        temp[c] = df[c].fillna("_absent_").astype(str)

    key = (
        temp["ensemble_size"]
        + "|c:" + temp["fam_cello"]
        + "|v:" + temp["fam_viola"]
        + "|v2:" + temp["fam_violin2"]
        + "|v1:" + temp["fam_violin1"]
    )
    return key

def _split_one_subset(df_all: pd.DataFrame, subset_name: str, seed: int):
    """
    Exact-size stratified split within one subset.
    Strategy:
      1. Build composite strata.
      2. Allocate exact test counts across strata.
      3. Allocate exact val counts on the remaining pool across strata.
      4. Train gets the rest.
    """
    mask = (df_all["subset"].values == subset_name)
    idx = np.flatnonzero(mask)
    df_sub = df_all.iloc[idx].copy()

    n_total = len(df_sub)
    if n_total == 0:
        raise RuntimeError(f"Subset '{subset_name}' is empty.")

    target_test = int(round(n_total * 0.15))
    target_val = int(round(n_total * 0.15))
    target_train = n_total - target_test - target_val

    strata = _build_strata_keys(df_sub)
    strata_counts = strata.value_counts(sort=False)
    strata_names = list(strata_counts.index)
    counts = strata_counts.values.astype(np.int64)

    # Exact test allocation per stratum
    test_alloc = _hamilton_allocate(counts, target_test)

    # Remaining after test
    remaining_after_test = counts - test_alloc

    # Exact val allocation per stratum on the remaining pool
    val_alloc = _hamilton_allocate(remaining_after_test, target_val)

    # Train gets the remainder
    train_alloc = remaining_after_test - val_alloc

    if train_alloc.sum() != target_train:
        raise RuntimeError(
            f"{subset_name}: train allocation mismatch {train_alloc.sum()} != {target_train}"
        )

    rng = np.random.RandomState(seed)

    train_parts, val_parts, test_parts = [], [], []

    # Precompute local positions per stratum
    local_positions = np.arange(n_total)
    strata_to_local = {}
    for s in strata_names:
        strata_to_local[s] = local_positions[strata.values == s]

    for i, s in enumerate(strata_names):
        loc = strata_to_local[s].copy()
        rng.shuffle(loc)

        n_te = int(test_alloc[i])
        n_va = int(val_alloc[i])
        n_tr = int(train_alloc[i])

        assert n_te + n_va + n_tr == len(loc)

        test_loc = loc[:n_te]
        val_loc = loc[n_te:n_te + n_va]
        train_loc = loc[n_te + n_va:]

        if len(train_loc) != n_tr:
            raise RuntimeError(f"{subset_name} / {s}: train slice mismatch")
        if len(val_loc) != n_va:
            raise RuntimeError(f"{subset_name} / {s}: val slice mismatch")
        if len(test_loc) != n_te:
            raise RuntimeError(f"{subset_name} / {s}: test slice mismatch")

        test_parts.append(idx[test_loc])
        val_parts.append(idx[val_loc])
        train_parts.append(idx[train_loc])

    train_idx = np.concatenate(train_parts) if train_parts else np.array([], dtype=np.int64)
    val_idx = np.concatenate(val_parts) if val_parts else np.array([], dtype=np.int64)
    test_idx = np.concatenate(test_parts) if test_parts else np.array([], dtype=np.int64)

    # Shuffle final arrays
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    # Final checks
    if len(train_idx) != target_train:
        raise RuntimeError(f"{subset_name}: final train size mismatch")
    if len(val_idx) != target_val:
        raise RuntimeError(f"{subset_name}: final val size mismatch")
    if len(test_idx) != target_test:
        raise RuntimeError(f"{subset_name}: final test size mismatch")

    return train_idx, val_idx, test_idx

# ---------------------------------------------------------------------
# Cache validation
# ---------------------------------------------------------------------
use_cache = False
if _split_file.exists() and _manifest_file.exists():
    manifest = pd.read_csv(_manifest_file)
    if len(manifest) == len(df_all):
        use_cache = True
    else:
        print("Cached splits incompatible — recomputing...")
        _split_file.unlink(missing_ok=True)
        _manifest_file.unlink(missing_ok=True)

if use_cache:
    print("Loading existing splits from disk (skipping recompute)...")
    _npz = np.load(_split_file)
    train_idx = _npz["train_idx"]
    val_idx   = _npz["val_idx"]
    test_idx  = _npz["test_idx"]

else:
    print("Computing exact-size stratified splits...")

    seed_base = SEED if "SEED" in globals() else 42

    tr_strict, va_strict, te_strict = _split_one_subset(df_all, "strict", seed_base + 1)
    print(
        f"  strict -> train: {len(tr_strict):,} ({len(tr_strict)/len(df_all[df_all['subset']=='strict']):.1%}) | "
        f"val: {len(va_strict):,} ({len(va_strict)/len(df_all[df_all['subset']=='strict']):.1%}) | "
        f"test: {len(te_strict):,} ({len(te_strict)/len(df_all[df_all['subset']=='strict']):.1%})"
    )

    tr_loose, va_loose, te_loose = _split_one_subset(df_all, "loose", seed_base + 2)
    print(
        f"  loose  -> train: {len(tr_loose):,} ({len(tr_loose)/len(df_all[df_all['subset']=='loose']):.1%}) | "
        f"val: {len(va_loose):,} ({len(va_loose)/len(df_all[df_all['subset']=='loose']):.1%}) | "
        f"test: {len(te_loose):,} ({len(te_loose)/len(df_all[df_all['subset']=='loose']):.1%})"
    )

    train_idx = np.concatenate([tr_strict, tr_loose])
    val_idx   = np.concatenate([va_strict, va_loose])
    test_idx  = np.concatenate([te_strict, te_loose])

    rng = np.random.RandomState(seed_base)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    # Integrity checks
    S_tr, S_va, S_te = set(train_idx), set(val_idx), set(test_idx)

    assert len(S_tr & S_va) == 0, "Train/Val overlap detected"
    assert len(S_tr & S_te) == 0, "Train/Test overlap detected"
    assert len(S_va & S_te) == 0, "Val/Test overlap detected"
    assert len(S_tr | S_va | S_te) == len(df_all), "Coverage mismatch detected"

    print("Disjointness & coverage: OK")

    np.savez_compressed(
        _split_file,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx
    )

    manifest = pd.DataFrame({
        "idx": np.arange(len(df_all)),
        "split": "none",
        "subset": df_all["subset"].values,
        "ensemble_size": df_all["ensemble_size"].values,
    })
    manifest.loc[train_idx, "split"] = "train"
    manifest.loc[val_idx,   "split"] = "val"
    manifest.loc[test_idx,  "split"] = "test"
    manifest.to_csv(_manifest_file, index=False)

    print("Saved splits")

# ---------------------------------------------------------------------
# Final reporting
# ---------------------------------------------------------------------
total = len(train_idx) + len(val_idx) + len(test_idx)
print(
    f"\nSplit sizes — "
    f"train: {len(train_idx):,} ({len(train_idx)/total:.1%}) | "
    f"val: {len(val_idx):,} ({len(val_idx)/total:.1%}) | "
    f"test: {len(test_idx):,} ({len(test_idx)/total:.1%})"
)

df_train = df_all.iloc[train_idx].reset_index(drop=True)
df_val   = df_all.iloc[val_idx].reset_index(drop=True)
df_test  = df_all.iloc[test_idx].reset_index(drop=True)

for name, df in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    print(f"\n{name} — subset x ensemble_size")
    print(pd.crosstab(df["subset"], df["ensemble_size"]))


## Cell 6b — Reload Splits from Disk
Run instead of Cell 5 on every session restart.


In [ ]:
# Cell 6b — Reload splits from disk (EXP4)

if "df_all" not in globals():
    raise RuntimeError("df_all is missing. Re-run Cells 3 → 4 first.")

OUT_SPLIT_DIR = Path(paths.split_dir)
NPZ_PATH = OUT_SPLIT_DIR / "exp4_split_indices.npz"
CSV_PATH = OUT_SPLIT_DIR / "exp4_manifest.csv"

for p in [NPZ_PATH, CSV_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Split file not found: {p}\nRun Cell 5 once.")

manifest = pd.read_csv(CSV_PATH)
if len(manifest) != len(df_all):
    raise RuntimeError("Split files incompatible with current df_all. Re-run Cell 5.")

data = np.load(NPZ_PATH)
train_idx = data["train_idx"]
val_idx   = data["val_idx"]
test_idx  = data["test_idx"]

df_train = df_all.iloc[train_idx].reset_index(drop=True)
df_val   = df_all.iloc[val_idx].reset_index(drop=True)
df_test  = df_all.iloc[test_idx].reset_index(drop=True)

total = len(train_idx) + len(val_idx) + len(test_idx)
print(f"Splits reloaded — "
      f"train: {len(df_train)} ({len(df_train)/total:.1%}) | "
      f"val: {len(df_val)} ({len(df_val)/total:.1%}) | "
      f"test: {len(df_test)} ({len(df_test)/total:.1%})")

s_tr = set(train_idx.tolist())
s_va = set(val_idx.tolist())
s_te = set(test_idx.tolist())
assert s_tr.isdisjoint(s_va) and s_tr.isdisjoint(s_te) and s_va.isdisjoint(s_te)
print("Disjointness: OK")

for name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    subsets = df["subset"].unique().tolist()
    assert "strict" in subsets and "loose" in subsets, \
        f"{name} split is missing a subset — re-run Cell 5."
print("Subset coverage: OK")

for name, df in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    print(f"\n{name} — subset x ensemble_size")
    print(pd.crosstab(df["subset"], df["ensemble_size"]))


## Cell 11.8 — Focus Technique-Family Selection (TRAIN only)
Defines `FOCUS_FAM` — the technique families used for classification.


In [ ]:
# Cell 11.8 — Focus technique family selection (TRAIN only)

if "df_train" not in globals():
    raise RuntimeError("df_train not found. Run previous cells first.")

# Count family occurrences across all instruments (train only)
fam_counter = Counter()
for inst in INSTR_ORDER:
    fam_col = f"fam_{inst}"
    fam_counter.update(df_train[fam_col].dropna().tolist())

total_labels = sum(fam_counter.values())
print("Family distribution (train, all instruments pooled):")
for fam in TECHNIQUE_FAMILIES:
    count = fam_counter.get(fam, 0)
    pct = 100 * count / total_labels if total_labels > 0 else 0
    print(f"  {fam:<12s} : {count:>8,}  ({pct:>5.1f}%)")

# Select focus families (>= 1% threshold)
MIN_PCT = 1.0
FOCUS_FAM = [
    fam for fam in TECHNIQUE_FAMILIES
    if 100 * fam_counter.get(fam, 0) / total_labels >= MIN_PCT
]

print(f"\nFocus families (>={MIN_PCT}%): {FOCUS_FAM}")

fam2i = {f: i for i, f in enumerate(FOCUS_FAM)}
N_FAM = len(FOCUS_FAM)

# Save
focus_path = os.path.join(paths.split_dir, "exp4_focus_families.json")
with open(focus_path, "w") as f:
    json.dump({"FOCUS_FAM": FOCUS_FAM, "fam2i": fam2i}, f, indent=2)
print(f"Saved: {focus_path}")


## Cell 11.9 — Focus Label Filtering
Keeps only samples where at least one present instrument has a family label in `FOCUS_FAM`.


In [ ]:
# Cell 11.9 — Focus label filtering (technique families)

assert "FOCUS_FAM" in globals(), "Run Cell 11.8 first."
assert all(x in globals() for x in ["df_train", "df_val", "df_test"]), \
    "Run earlier cells first."

def _row_has_focus_fam(row):
    for inst in INSTR_ORDER:
        col = f"fam_{inst}"
        if col in row and pd.notna(row[col]) and str(row[col]) in FOCUS_FAM:
            return True
    return False

for name in ["df_train", "df_val", "df_test"]:
    df = globals()[name]
    before = len(df)
    df_filtered = df[df.apply(_row_has_focus_fam, axis=1)].reset_index(drop=True)
    globals()[name] = df_filtered
    after = len(df_filtered)
    print(f"  {name}: {before:,} → {after:,} ({after/before:.1%})")

print("Focus family filtering: OK")


## Cell 11.10 — Class Weights for Imbalance Correction
Computes effective-number class weights for the selected technique families.


In [ ]:
# Cell 11.10 — Class weights (effective number; technique families)

assert "FOCUS_FAM" in globals(), "Run Cell 11.8 first."

# Count per-family occurrences (train only, all instruments pooled)
# Vectorized counting (fast)
fam_counts = np.zeros(N_FAM, dtype=np.float64)

for inst in INSTR_ORDER:
    fam_col = f"fam_{inst}"
    vals = df_train[fam_col].dropna().values
    for fam, count in pd.Series(vals).value_counts().items():
        if fam in fam2i:
            fam_counts[fam2i[fam]] += count

print("Family counts (train):")
for i, fam in enumerate(FOCUS_FAM):
    print(f"  {fam:<12s} : {int(fam_counts[i]):>8,}")

# Effective-number weighting
beta = 0.9999
effective_num = 1.0 - np.power(beta, fam_counts)
weights = (1.0 - beta) / (effective_num + 1e-8)
weights = weights / weights.sum() * N_FAM

CLASS_WEIGHTS_FAM = weights.astype(np.float32)

print("\nClass weights:")
for i, fam in enumerate(FOCUS_FAM):
    print(f"  {fam:<12s} : {CLASS_WEIGHTS_FAM[i]:.4f}")

# Save
weights_path = os.path.join(paths.split_dir, "exp4_class_weights_fam.json")
with open(weights_path, "w") as f:
    json.dump({
        "FOCUS_FAM": FOCUS_FAM,
        "CLASS_WEIGHTS_FAM": CLASS_WEIGHTS_FAM.tolist(),
        "family_counts": fam_counts.tolist(),
    }, f, indent=2)
print(f"\nSaved: {weights_path}")


## Cell 11.11 — Dataset, Collate, and Split Construction (EXP4)
Defines `SECDFamilyDataset` that returns `input_values` (128×256 mel) and
`y_fam` (per-instrument technique-family labels). Training uses masked
loss so absent instruments do not contribute to the objective.


In [ ]:
# Cell 11.11 — Dataset construction (technique families)

for _n in ("df_train", "df_val", "df_test", "FOCUS_FAM", "INSTR_ORDER"):
    if _n not in globals():
        raise RuntimeError(f"{_n} missing — run previous cells.")

TARGET_FRAMES = 256

def pad_or_truncate(mel, n_frames):
    t = mel.shape[-1]
    if t >= n_frames:
        return mel[..., :n_frames]
    return torch.nn.functional.pad(mel, (0, n_frames - t))

fam_labels = FOCUS_FAM
N_FAM = len(fam_labels)

print("Technique family classes:", fam_labels)

class SECDFamilyDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def _lab(self, row, inst):
        col = f"fam_{inst}"
        if col not in row or row[col] is None:
            return -1
        return fam2i.get(str(row[col]), -1)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        try:
            mel = torch.tensor(np.load(row["cachefile"]).astype(np.float32))
        except Exception:
            mel = torch.zeros(128, TARGET_FRAMES)

        mel = pad_or_truncate(mel, TARGET_FRAMES)

        y_fam = torch.tensor(
            [self._lab(row, inst) for inst in INSTR_ORDER],
            dtype=torch.long
        )

        return {
            "input_values": mel,
            "y_fam": y_fam,
        }

def collate_fn(batch):
    return {
        "input_values": torch.stack([b["input_values"] for b in batch]),
        "y_fam": torch.stack([b["y_fam"] for b in batch]),
    }

ds_train = SECDFamilyDataset(df_train)
ds_val   = SECDFamilyDataset(df_val)
ds_test  = SECDFamilyDataset(df_test)

print(f"Train : {len(ds_train):,}")
print(f"Val   : {len(ds_val):,}")
print(f"Test  : {len(ds_test):,}")


## Cell 12 — Full Training Pipeline (EXP4 / technique-family classification)
Defines and trains the AST-based technique-family model with per-instrument
heads and masked loss over family targets.
**This cell is fully self-contained** — `resize_ast_pos_embed` and `masked_ce` are defined inline.


In [ ]:
# Cell 12 — Full Training Pipeline (Technique Families, EXP4)

# ------------------------------------------------------------------
# Environment
# ------------------------------------------------------------------
warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Patch: PyTorch 2.6 Trainer RNG loading issue
_orig_load_rng = getattr(_hf_trainer.Trainer, "_load_rng_state", None)
if _orig_load_rng is not None:
    def _patched_load_rng(self, checkpoint):
        rng_file = os.path.join(checkpoint, f"rng_state_{self.args.process_index}.pth")
        if os.path.isfile(rng_file):
            try:
                with open(rng_file, "rb") as f:
                    buf = io.BytesIO(f.read())
                state = torch.load(buf, map_location="cpu", weights_only=False)
                random.setstate(state["python"])
                np.random.set_state(state["numpy"])
                torch.random.set_rng_state(state["cpu"])
                if "cuda" in state and torch.cuda.is_available():
                    torch.cuda.random.set_rng_state_all(state["cuda"])
            except Exception as e:
                print(f"  Warning: RNG reload skipped: {e}")
    _hf_trainer.Trainer._load_rng_state = _patched_load_rng

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------------
# Guards
# ------------------------------------------------------------------
need = [
    "INSTR_ORDER", "TARGET_FRAMES", "FOCUS_FAM", "CLASS_WEIGHTS_FAM",
    "ds_train", "ds_val", "ds_test", "collate_fn",
    "MODEL_NAME", "N_FAM", "SEED", "paths"
]
miss = [k for k in need if k not in globals()]
if miss:
    raise RuntimeError(f"Missing from previous cells: {miss}")

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found.")

print(f"  GPU : {torch.cuda.get_device_name(0)}")

_disk = shutil.disk_usage(paths.base)
_free_gb = _disk.free / 1024**3
if _free_gb < 5.0:
    raise RuntimeError(f"Only {_free_gb:.1f} GB free — need at least 5 GB.")
print(f"  Disk: {_free_gb:.1f} GB free")

# ------------------------------------------------------------------
# Run directory
# ------------------------------------------------------------------
RUN_TAG = time.strftime("exp4_tecfam_%Y%m%d-%H%M%S")
OUTDIR = os.path.join(paths.outdir, RUN_TAG)
os.makedirs(OUTDIR, exist_ok=True)

with open(os.path.join(paths.outdir, "latest_run.txt"), "w") as f:
    f.write(OUTDIR)

print(f"  Run  : {OUTDIR}")

# ------------------------------------------------------------------
# Save session config
# ------------------------------------------------------------------
W_FAM = np.asarray(CLASS_WEIGHTS_FAM, dtype=np.float32)

print(f"\n  Families  : {N_FAM} — {FOCUS_FAM}")
print(f"  Weights   : {list(W_FAM)}")

session_cfg = {
    "experiment": "EXP4_TEC_FAM",
    "MODEL_NAME": MODEL_NAME,
    "INSTR_ORDER": INSTR_ORDER,
    "FOCUS_FAM": FOCUS_FAM,
    "N_FAM": N_FAM,
    "CLASS_WEIGHTS_FAM": W_FAM.tolist(),
    "TARGET_FRAMES": TARGET_FRAMES,
    "SEED": SEED,
    "train_size": len(ds_train),
    "val_size": len(ds_val),
    "test_size": len(ds_test),
}
with open(os.path.join(OUTDIR, "session_config.json"), "w") as f:
    json.dump(session_cfg, f, indent=2)

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def resize_ast_pos_embed(ast_model, target_frames: int):
    """Interpolate AST positional embeddings to match target frame count."""
    n_mels_patches = (128 - 16) // 10 + 1
    n_time_patches = (target_frames - 16) // 10 + 1
    n_patches = n_mels_patches * n_time_patches
    n_tokens = n_patches + 2  # CLS + distillation

    pos_embed = ast_model.embeddings.position_embeddings
    old_n = pos_embed.shape[1]

    if old_n == n_tokens:
        return

    cls_dist = pos_embed[:, :2, :]
    patch_pos = pos_embed[:, 2:, :]

    old_freq = 12
    old_time = 101
    d = patch_pos.shape[-1]

    patch_pos = patch_pos.reshape(1, old_freq, old_time, d).permute(0, 3, 1, 2)
    patch_pos = F.interpolate(
        patch_pos.float(),
        size=(n_mels_patches, n_time_patches),
        mode="bicubic",
        align_corners=False,
    )
    patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, n_patches, d)

    new_pos = torch.cat([cls_dist, patch_pos], dim=1)
    ast_model.embeddings.position_embeddings = nn.Parameter(new_pos)

    print(
        f"  Pos embeddings resized: {old_n} → {n_tokens} "
        f"(grid {old_freq}×{old_time} → {n_mels_patches}×{n_time_patches})"
    )

def masked_ce(logits_flat, targets_flat, weight=None, ignore_index=-1):
    valid = targets_flat.ne(ignore_index)
    if not valid.any():
        return logits_flat.new_zeros(())
    return F.cross_entropy(
        logits_flat[valid].float(),
        targets_flat[valid],
        weight=weight.float() if weight is not None else None,
        label_smoothing=0.05,
    )

# ------------------------------------------------------------------
# Model
# ------------------------------------------------------------------
class ASTMultiHeadFamilies(nn.Module):
    def __init__(self, model_name, instr_order, n_fam, class_weights):
        super().__init__()
        self.n_instr = len(instr_order)
        self.n_fam = n_fam

        self.ast = AutoModel.from_pretrained(model_name, ignore_mismatched_sizes=True)
        resize_ast_pos_embed(self.ast, TARGET_FRAMES)

        with torch.no_grad():
            dummy = torch.zeros(1, 128, TARGET_FRAMES)
            h = self.ast(input_values=dummy).last_hidden_state
            d = h.mean(dim=1).shape[-1]

        self.inst_embed = nn.Embedding(self.n_instr, d)

        self.heads_fam = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d, d),
                nn.GELU(),
                nn.Linear(d, n_fam)
            )
            for _ in range(self.n_instr)
        ])

        self.register_buffer("w_fam", torch.tensor(class_weights, dtype=torch.float32))

    def feats(self, x):
        return self.ast(input_values=x).last_hidden_state.mean(dim=1)

    def forward(self, input_values, y_fam=None, **kwargs):
        emb = self.feats(input_values)

        inst_ids = torch.arange(self.n_instr, device=emb.device)
        inst_vecs = self.inst_embed(inst_ids)

        logits_list = []
        for i, head in enumerate(self.heads_fam):
            inst_vec = inst_vecs[i].unsqueeze(0).expand_as(emb)
            conditioned = emb + inst_vec
            logits_list.append(head(conditioned))

        logits = torch.stack(logits_list, dim=1)  # [B, 4, N_FAM]

        loss = None
        if y_fam is not None:
            loss = masked_ce(
                logits.reshape(-1, self.n_fam),
                y_fam.reshape(-1),
                weight=self.w_fam
            )

        return {"loss": loss, "logits": logits}

# ------------------------------------------------------------------
# Metrics
# ------------------------------------------------------------------
def compute_metrics_fam(eval_pred):
    preds, labels = eval_pred
    logits = np.asarray(preds)
    y_true = np.asarray(labels)

    all_preds, all_labels = [], []
    per_inst = {}

    for i, inst in enumerate(INSTR_ORDER):
        inst_logits = logits[:, i, :]
        inst_labels = y_true[:, i]

        mask = inst_labels >= 0
        if mask.sum() == 0:
            continue

        yp = inst_logits[mask].argmax(axis=-1)
        yt = inst_labels[mask]

        all_preds.extend(yp.tolist())
        all_labels.extend(yt.tolist())

        per_inst[inst] = {
            "acc": float(accuracy_score(yt, yp)),
            "f1": float(f1_score(yt, yp, average="macro", zero_division=0)),
            "n": int(mask.sum()),
        }

    pooled_acc = float(accuracy_score(all_labels, all_preds)) if all_labels else 0.0
    pooled_f1 = float(f1_score(all_labels, all_preds, average="macro", zero_division=0)) if all_labels else 0.0

    out = {
        "eval_support_fam": len(all_labels),
        "eval_acc_fam": pooled_acc,
        "eval_f1_macro_fam": pooled_f1,
    }

    for inst, m in per_inst.items():
        out[f"eval_acc_{inst}"] = m["acc"]
        out[f"eval_f1_{inst}"] = m["f1"]
        out[f"eval_n_{inst}"] = m["n"]

    return out

# ------------------------------------------------------------------
# Callback
# ------------------------------------------------------------------
class EpochTableCallback(TrainerCallback):
    _SEP = "-" * 140
    _HDR = (
        f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>8} | "
        f"{'Acc':>7} | {'F1':>7} | "
        f"{'F1_cello':>9} | {'F1_viola':>9} | {'F1_violin2':>11} | {'F1_violin1':>11}"
    )

    def __init__(self):
        self.train_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.train_losses.append(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return

        if state.epoch == 1:
            print(self._SEP)
            print(self._HDR)
            print(self._SEP)

        ep = int(state.epoch or 0)
        tl = float(np.mean(self.train_losses)) if self.train_losses else 0.0
        self.train_losses = []

        vl = metrics.get("eval_loss", 0.0)
        acc = metrics.get("eval_acc_fam", 0.0)
        f1 = metrics.get("eval_f1_macro_fam", 0.0)

        f1_cello   = metrics.get("eval_f1_cello", 0.0)
        f1_viola   = metrics.get("eval_f1_viola", 0.0)
        f1_violin2 = metrics.get("eval_f1_violin2", 0.0)
        f1_violin1 = metrics.get("eval_f1_violin1", 0.0)

        print(
            f"{ep:>5} | {tl:>10.4f} | {vl:>8.4f} | "
            f"{acc:>7.4f} | {f1:>7.4f} | "
            f"{f1_cello:>9.4f} | {f1_viola:>9.4f} | "
            f"{f1_violin2:>11.4f} | {f1_violin1:>11.4f}"
        )

# ------------------------------------------------------------------
# CPU dry-run
# ------------------------------------------------------------------
print("\n  CPU dry-run...")
s0 = ds_train[0]
s1 = ds_train[1]
batch = collate_fn([s0, s1])

m_cpu = ASTMultiHeadFamilies(MODEL_NAME, INSTR_ORDER, N_FAM, W_FAM)
out = m_cpu(input_values=batch["input_values"].float(), y_fam=batch["y_fam"].long())
assert out["loss"] is not None
assert out["logits"].shape == (2, len(INSTR_ORDER), N_FAM)
_ = float(out["loss"])
print("  CPU dry-run passed.")
del m_cpu
torch.cuda.empty_cache()

# ------------------------------------------------------------------
# CUDA model
# ------------------------------------------------------------------
device = torch.device("cuda:0")
model = ASTMultiHeadFamilies(MODEL_NAME, INSTR_ORDER, N_FAM, W_FAM).to(device)

try:
    model.ast.gradient_checkpointing_enable()
    print("  Gradient checkpointing : ENABLED")
except Exception as e:
    print(f"  Gradient checkpointing : unavailable ({e})")

# ------------------------------------------------------------------
# TrainingArguments
# ------------------------------------------------------------------
args = TrainingArguments(
    output_dir=os.path.join(OUTDIR, "hf_ckpt"),
    logging_dir=os.path.join(OUTDIR, "logs"),

    num_train_epochs=40,
    per_device_train_batch_size=384,
    per_device_eval_batch_size=512,

    learning_rate=3e-5,
    warmup_ratio=0.05,
    weight_decay=1e-2,
    bf16=True,
    dataloader_num_workers=8,
    dataloader_pin_memory=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro_fam",
    greater_is_better=True,
    save_total_limit=2,

    remove_unused_columns=False,
    label_names=["y_fam"],
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collate_fn,
    compute_metrics=compute_metrics_fam,
    callbacks=[
        EpochTableCallback(),
        EarlyStoppingCallback(
            early_stopping_patience=5,
            early_stopping_threshold=0.001,
        ),
    ],
)

trainer.remove_callback(PrinterCallback)

print("-" * 90)
print(f"Run  : {OUTDIR}")
print(f"Model: {MODEL_NAME}")
print(f"Task : {N_FAM}-class technique family classification")
print(f"Data : train={len(ds_train):,} val={len(ds_val):,} test={len(ds_test):,}")
print("-" * 90)

# ------------------------------------------------------------------
# Train
# ------------------------------------------------------------------
t0 = time.time()
trainer.train()

print(f"\nTraining finished in {(time.time() - t0)/60:.2f} min")

trainer.save_model(os.path.join(args.output_dir, "final"))
trainer.save_state()

print("-" * 90)
print("FULL RUN COMPLETE — EXP4")
print(f"Artifacts -> {OUTDIR}")
print("-" * 90)


## Cell 13 — Post-Training Evaluation and Artifacts (EXP4 technique-family task)
Loads the best checkpoint and runs inference on val/test sets.
**Self-contained** — `resize_ast_pos_embed` and `masked_ce` are defined inline.


In [ ]:
# Cell 13 — Post-Training Evaluation and Artifacts (EXP4 technique-family task)

matplotlib.use("Agg")

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Self-contained helpers ──────────────────────────────────

def resize_ast_pos_embed(ast_model, target_frames: int):
    n_mels_patches = (128 - 16) // 10 + 1
    n_time_patches = (target_frames - 16) // 10 + 1
    n_patches = n_mels_patches * n_time_patches
    n_tokens = n_patches + 2
    pos_embed = ast_model.embeddings.position_embeddings
    old_n = pos_embed.shape[1]
    if old_n == n_tokens:
        return
    cls_dist = pos_embed[:, :2, :]
    patch_pos = pos_embed[:, 2:, :]
    old_freq, old_time, d = 12, 101, patch_pos.shape[-1]
    patch_pos = patch_pos.reshape(1, old_freq, old_time, d).permute(0, 3, 1, 2)
    patch_pos = F.interpolate(
        patch_pos.float(),
        size=(n_mels_patches, n_time_patches),
        mode="bicubic",
        align_corners=False,
    )
    patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, n_patches, d)
    new_pos = torch.cat([cls_dist, patch_pos], dim=1)
    ast_model.embeddings.position_embeddings = nn.Parameter(new_pos)

def masked_ce(logits, targets, weight=None, ignore_index=-1):
    valid = targets.ne(ignore_index)
    if not valid.any():
        return logits.new_zeros(())
    return F.cross_entropy(
        logits[valid].float(),
        targets[valid],
        weight=weight.float() if weight is not None else None,
        label_smoothing=0.05,
    )

BASE = paths.outdir

if "OUTDIR" not in globals() or not os.path.isdir(globals().get("OUTDIR", "")):
    ptr = os.path.join(BASE, "latest_run.txt")
    if not os.path.exists(ptr):
        raise RuntimeError("OUTDIR not found and latest_run.txt is missing.")
    with open(ptr, "r", encoding="utf-8") as f:
        OUTDIR = f.read().strip()
    print(f"Loaded run path -> {OUTDIR}")
else:
    print(f"Using OUTDIR -> {OUTDIR}")

CKPT_DIR = os.path.join(OUTDIR, "hf_ckpt")
ARTIFACT_DIR = os.path.join(OUTDIR, "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

cfg_path = os.path.join(OUTDIR, "session_config.json")
if not os.path.exists(cfg_path):
    raise RuntimeError("Missing session_config.json")

with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)

FOCUS_FAM = list(cfg["FOCUS_FAM"])
TARGET_FRAMES = int(cfg["TARGET_FRAMES"])
MODEL_NAME = cfg["MODEL_NAME"]
W_FAM = np.array(cfg["CLASS_WEIGHTS_FAM"], dtype=np.float32)
INSTR_ORDER = list(cfg["INSTR_ORDER"])
N_FAM = int(cfg["N_FAM"])

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device -> {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

need = ["ds_val", "ds_test", "collate_fn"]
for req in need:
    if req not in globals():
        raise RuntimeError(f"Missing: {req}")

print(f"VAL: {len(ds_val):,} | TEST: {len(ds_test):,}")

class ASTMultiHeadFamilies(nn.Module):
    def __init__(self, model_name, instr_order, n_fam, class_weights):
        super().__init__()
        self.n_instr = len(instr_order)
        self.n_fam = n_fam

        self.ast = AutoModel.from_pretrained(model_name, ignore_mismatched_sizes=True)
        resize_ast_pos_embed(self.ast, TARGET_FRAMES)

        with torch.no_grad():
            dummy = torch.zeros(1, 128, TARGET_FRAMES)
            h = self.ast(input_values=dummy).last_hidden_state
            d = h.mean(dim=1).shape[-1]

        self.inst_embed = nn.Embedding(self.n_instr, d)

        self.heads_fam = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d, d),
                nn.GELU(),
                nn.Linear(d, n_fam),
            )
            for _ in range(self.n_instr)
        ])

        self.register_buffer("w_fam", torch.tensor(class_weights, dtype=torch.float32))

    def feats(self, x):
        return self.ast(input_values=x).last_hidden_state.mean(dim=1)

    def forward(self, input_values, y_fam=None, **kwargs):
        emb = self.feats(input_values)

        inst_ids = torch.arange(self.n_instr, device=emb.device)
        inst_vecs = self.inst_embed(inst_ids)

        logits_list = []
        for i, head in enumerate(self.heads_fam):
            inst_vec = inst_vecs[i].unsqueeze(0).expand_as(emb)
            conditioned = emb + inst_vec
            logits_list.append(head(conditioned))

        logits = torch.stack(logits_list, dim=1)

        loss = None
        if y_fam is not None:
            loss = masked_ce(
                logits.reshape(-1, self.n_fam),
                y_fam.reshape(-1),
                weight=self.w_fam,
            )
        return {"loss": loss, "logits": logits}

def resolve_best_checkpoint(ckpt_dir):
    state_path = os.path.join(ckpt_dir, "trainer_state.json")
    if not os.path.exists(state_path):
        return None
    with open(state_path, "r", encoding="utf-8") as f:
        state = json.load(f)
    best = state.get("best_model_checkpoint")
    if best and os.path.isdir(best):
        print(f"BEST checkpoint -> {best}")
        return best
    return None

model_dir = resolve_best_checkpoint(CKPT_DIR)

if model_dir is None:
    checkpoints = sorted(
        glob.glob(os.path.join(CKPT_DIR, "checkpoint-*")),
        key=lambda x: int(x.split("-")[-1])
    )
    if not checkpoints:
        raise RuntimeError("No checkpoints found")
    model_dir = checkpoints[-1]
    print(f"Fallback -> {model_dir}")

model = ASTMultiHeadFamilies(MODEL_NAME, INSTR_ORDER, N_FAM, W_FAM)

sf_path = os.path.join(model_dir, "model.safetensors")
pt_path = os.path.join(model_dir, "pytorch_model.bin")

if os.path.exists(sf_path):
    state_dict = st.load_file(sf_path, device=str(device))
elif os.path.exists(pt_path):
    state_dict = torch.load(pt_path, map_location=device)
else:
    raise RuntimeError(f"No checkpoint weights found inside: {model_dir}")

load_info = model.load_state_dict(state_dict, strict=False)
if load_info.missing_keys:
    print(f"Missing keys    : {load_info.missing_keys}")
if load_info.unexpected_keys:
    print(f"Unexpected keys : {load_info.unexpected_keys}")

model.to(device)
model.eval()

eval_trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=ARTIFACT_DIR,
        per_device_eval_batch_size=512,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        report_to=[],
        remove_unused_columns=False,
        label_names=["y_fam"],
        use_cpu=(device.type == "cpu"),
        bf16=(device.type == "cuda"),
        seed=SEED,
    ),
    data_collator=collate_fn,
)

def run_predictions(dataset):
    pred = eval_trainer.predict(dataset)
    raw_logits = pred.predictions
    raw_labels = pred.label_ids

    logits = np.asarray(raw_logits[0]) if isinstance(raw_logits, tuple) else np.asarray(raw_logits)
    labels = np.asarray(raw_labels[0]) if isinstance(raw_labels, tuple) else np.asarray(raw_labels)

    return logits, labels

def save_cm(y_true, y_pred, labels, title, path, normalize=None):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(labels))),
        normalize=normalize
    )
    fig, ax = plt.subplots(figsize=(7, 6))
    fmt = ".2f" if normalize is not None else "d"
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(
        ax=ax,
        cmap="Blues",
        colorbar=False,
        values_format=fmt
    )
    ax.set_title(title, fontsize=12, pad=10)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

all_metrics = {}

for split_name, dataset in [("val", ds_val), ("test", ds_test)]:
    print(f"\n=== {split_name.upper()} ===")

    logits_raw, labels_raw = run_predictions(dataset)

    pooled_preds = []
    pooled_true = []

    split_metrics = {"per_instrument": {}, "pooled": None}

    for i, inst in enumerate(INSTR_ORDER):
        inst_logits = logits_raw[:, i, :]
        inst_labels = labels_raw[:, i]

        mask = inst_labels >= 0
        if mask.sum() == 0:
            continue

        y_true = inst_labels[mask]
        y_pred = inst_logits[mask].argmax(axis=-1)

        pooled_true.extend(y_true.tolist())
        pooled_preds.extend(y_pred.tolist())

        rep = classification_report(
            y_true,
            y_pred,
            labels=list(range(N_FAM)),
            target_names=FOCUS_FAM,
            output_dict=True,
            zero_division=0,
        )

        split_metrics["per_instrument"][inst] = rep

        print(
            f"{split_name} | {inst} | "
            f"acc={rep['accuracy']:.4f} | "
            f"f1={rep['macro avg']['f1-score']:.4f} | "
            f"n={len(y_true):,}"
        )

        save_cm(
            y_true, y_pred, FOCUS_FAM,
            f"{split_name.upper()} | {inst} | Technique family — Absolute",
            os.path.join(ARTIFACT_DIR, f"cm_{split_name}_fam_{inst}_abs.png"),
            normalize=None,
        )
        save_cm(
            y_true, y_pred, FOCUS_FAM,
            f"{split_name.upper()} | {inst} | Technique family — Normalized",
            os.path.join(ARTIFACT_DIR, f"cm_{split_name}_fam_{inst}_norm.png"),
            normalize="true",
        )

    pooled_true = np.asarray(pooled_true)
    pooled_preds = np.asarray(pooled_preds)

    pooled_rep = classification_report(
        pooled_true,
        pooled_preds,
        labels=list(range(N_FAM)),
        target_names=FOCUS_FAM,
        output_dict=True,
        zero_division=0,
    )

    split_metrics["pooled"] = pooled_rep
    all_metrics[split_name] = split_metrics

    print(
        f"{split_name} | pooled | "
        f"acc={pooled_rep['accuracy']:.4f} | "
        f"f1={pooled_rep['macro avg']['f1-score']:.4f}"
    )

    save_cm(
        pooled_true, pooled_preds, FOCUS_FAM,
        f"{split_name.upper()} | Technique family — Pooled Absolute",
        os.path.join(ARTIFACT_DIR, f"cm_{split_name}_fam_pooled_abs.png"),
        normalize=None,
    )
    save_cm(
        pooled_true, pooled_preds, FOCUS_FAM,
        f"{split_name.upper()} | Technique family — Pooled Normalized",
        os.path.join(ARTIFACT_DIR, f"cm_{split_name}_fam_pooled_norm.png"),
        normalize="true",
    )

with open(os.path.join(ARTIFACT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2)

pngs = sorted(f for f in os.listdir(ARTIFACT_DIR) if f.endswith(".png"))
print(f"\nArtifacts saved -> {ARTIFACT_DIR}")
print(f"metrics.json + {len(pngs)} PNGs")
for p in pngs:
    print(f"  {p}")


## Cell 13.05 — Training Curves (Loss + F1)

Reads `trainer_state.json` from the HF checkpoint directory and parses `log_history` to extract per-epoch train loss, validation loss, validation accuracy, and validation macro F1. Produces two PNGs saved to `OUTDIR/artifacts/`: `loss_curves.png` (train vs val loss) and `f1_curves.png` (val accuracy + macro F1 over epochs). The F1 plot is skipped gracefully if the log history contains neither `eval_acc_fam` nor `eval_f1_macro_fam` entries.


In [ ]:
# Cell 13.05 — Training curve plots (EXP4)

matplotlib.use("Agg")

ckpt_dir = os.path.join(OUTDIR, "hf_ckpt")
art_dir = os.path.join(OUTDIR, "artifacts")
os.makedirs(art_dir, exist_ok=True)

state_path = os.path.join(ckpt_dir, "trainer_state.json")
if not os.path.exists(state_path):
    print("trainer_state.json not found — skipping curves")
else:
    with open(state_path, "r", encoding="utf-8") as f:
        state = json.load(f)

    log_history = state.get("log_history", [])

    train_loss = [(e.get("epoch"), e["loss"]) for e in log_history if "loss" in e and "eval_loss" not in e]
    eval_entries = [e for e in log_history if "eval_loss" in e]
    eval_loss = [(e.get("epoch"), e["eval_loss"]) for e in eval_entries]
    eval_f1 = [(e.get("epoch"), e.get("eval_f1_macro_fam", 0.0)) for e in eval_entries]
    eval_acc = [(e.get("epoch"), e.get("eval_acc_fam", 0.0)) for e in eval_entries]

    # ------------------------------------------------------------
    # Loss curves
    # ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(7, 5))
    if train_loss:
        ax.plot(*zip(*train_loss), label="Train loss")
    if eval_loss:
        ax.plot(*zip(*eval_loss), label="Val loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("EXP4 — Loss Curves")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    loss_path = os.path.join(art_dir, "loss_curves.png")
    fig.savefig(loss_path, dpi=150)
    plt.close(fig)

    # ------------------------------------------------------------
    # F1 curves
    # ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(7, 5))
    if eval_f1:
        ax.plot(*zip(*eval_f1), marker="o", markersize=3)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Macro F1")
    ax.set_title("EXP4 — Validation Macro F1")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    f1_path = os.path.join(art_dir, "f1_curves.png")
    fig.savefig(f1_path, dpi=150)
    plt.close(fig)

    # ------------------------------------------------------------
    # Accuracy curves
    # ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(7, 5))
    if eval_acc:
        ax.plot(*zip(*eval_acc), marker="o", markersize=3)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title("EXP4 — Validation Accuracy")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    acc_path = os.path.join(art_dir, "acc_curves.png")
    fig.savefig(acc_path, dpi=150)
    plt.close(fig)

    print(f"Saved: {loss_path}")
    print(f"Saved: {f1_path}")
    print(f"Saved: {acc_path}")


## Cell 13.1 — Artifact Preview (EXP4 technique-family task)

Displays all PNG artifacts produced by Cells 13 and 13.05 inline in the notebook using `IPython.display`. The images are shown in a fixed order: training curves, pooled confusion matrices, then per-instrument confusion matrices. Missing files are listed at the end rather than raising an error, so the cell can be run even if only a subset of plots was generated.


In [ ]:
# Cell 13.1 — Artifact Preview (EXP4 technique-family task)

art_dir = os.path.join(OUTDIR, "artifacts")

if not os.path.isdir(art_dir):
    print(f"Artifact directory not found: {art_dir}")
else:
    ordered_pngs = [
        "loss_curves.png",
        "f1_curves.png",
        "acc_curves.png",
        "cm_val_fam_pooled_abs.png",
        "cm_val_fam_pooled_norm.png",
        "cm_test_fam_pooled_abs.png",
        "cm_test_fam_pooled_norm.png",
    ]

    for inst in INSTR_ORDER:
        for split in ["val", "test"]:
            for suffix in ["abs", "norm"]:
                ordered_pngs.append(f"cm_{split}_fam_{inst}_{suffix}.png")

    missing = []

    print(f"Artifact Preview — {art_dir}\n")

    for fname in ordered_pngs:
        fpath = os.path.join(art_dir, fname)
        if os.path.exists(fpath):
            print(f"  {fname}")
            display(IPImage(filename=fpath, width=900))
            print()
        else:
            missing.append(fname)

    if missing:
        print(f"Missing artifacts ({len(missing)}):")
        for f in missing:
            print(f"  - {f}")
    else:
        print(f"All {len(ordered_pngs)} expected artifacts present.")
